# Consultas Analíticas
En esta sección se realizan consultas SQL sobre el modelo dimensional construido en la capa Gold.

Las consultas permiten obtener indicadores académicos y financieros que servirán como base para la creación de dashboards.

In [0]:
fact = spark.table(f"rendimiento_estudiantil.gold.fact_rendimiento_academico")

dim_estudiantes = spark.table(f"rendimiento_estudiantil.gold.dim_estudiantes")

dim_cursos = spark.table(f"rendimiento_estudiantil.gold.dim_cursos")

dim_semestres = spark.table(f"rendimiento_estudiantil.gold.dim_semestres")

In [0]:
fact.createOrReplaceTempView("fact_rendimiento_academico")

dim_estudiantes.createOrReplaceTempView("dim_estudiantes")

dim_cursos.createOrReplaceTempView("dim_cursos")

dim_semestres.createOrReplaceTempView("dim_semestres")

## 1. Preguntas de negocio

¿Cuál es la situación actual del rendimiento académico y del riesgo potencial de abandono?

In [0]:
%sql
SELECT
Nivel_Riesgo,
COUNT(*) AS Total_Estudiantes
FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes
GROUP BY Nivel_Riesgo;

2. ¿Qué factores personales y académicos están asociados al riesgo?

Riesgo por género

In [0]:
%sql
SELECT

Gender,

SUM(CASE WHEN Nivel_Riesgo = 'Alto' THEN 1 ELSE 0 END) AS Alto,

SUM(CASE WHEN Nivel_Riesgo = 'Medio' THEN 1 ELSE 0 END) AS Medio,

SUM(CASE WHEN Nivel_Riesgo = 'Bajo' THEN 1 ELSE 0 END) AS Bajo,

COUNT(*) AS Total

FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes

GROUP BY Gender

ORDER BY Gender;

Riesgo por carrera

In [0]:
%sql
SELECT

Major_Subject,

SUM(CASE WHEN Nivel_Riesgo = 'Alto' THEN 1 ELSE 0 END) AS Alto,

SUM(CASE WHEN Nivel_Riesgo = 'Medio' THEN 1 ELSE 0 END) AS Medio,

SUM(CASE WHEN Nivel_Riesgo = 'Bajo' THEN 1 ELSE 0 END) AS Bajo,

COUNT(*) AS Total

FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes

GROUP BY

Major_Subject

ORDER BY Major_Subject;

Riesgo por ingresos

In [0]:
%sql
SELECT

Family_Income_Level,

SUM(CASE WHEN Nivel_Riesgo = 'Alto' THEN 1 ELSE 0 END) AS Alto,

SUM(CASE WHEN Nivel_Riesgo = 'Medio' THEN 1 ELSE 0 END) AS Medio,

SUM(CASE WHEN Nivel_Riesgo = 'Bajo' THEN 1 ELSE 0 END) AS Bajo,

COUNT(*) AS Total

FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes

GROUP BY

Family_Income_Level

ORDER BY Family_Income_Level;

Riesgo por Región

In [0]:
%sql
SELECT

Region_Type,

SUM(CASE WHEN Nivel_Riesgo = 'Alto' THEN 1 ELSE 0 END) AS Alto,

SUM(CASE WHEN Nivel_Riesgo = 'Medio' THEN 1 ELSE 0 END) AS Medio,

SUM(CASE WHEN Nivel_Riesgo = 'Bajo' THEN 1 ELSE 0 END) AS Bajo,

COUNT(*) AS Total

FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes

GROUP BY

Region_Type

ORDER BY Region_Type;

3. ¿Qué carreras y cursos presentan mayor concentración de estudiantes con bajo rendimiento?

Carrera

In [0]:
%sql
SELECT

Major_Subject,

ROUND(AVG(Promedio_Final),2) Promedio

FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes

GROUP BY Major_Subject

ORDER BY Promedio;

Cursos

In [0]:
%sql
SELECT

nombre_curso,

ROUND(AVG(nota_final),2) Promedio

FROM rendimiento_estudiantil.gold.vw_monitoreo_riesgo

GROUP BY nombre_curso

ORDER BY Promedio;

3. Promedio por región

In [0]:
%sql
SELECT
  Region_Type,
  ROUND(AVG(nota_final),2) AS Promedio
FROM rendimiento_estudiantil.gold.vw_monitoreo_riesgo
GROUP BY Region_Type
ORDER BY Promedio DESC;

4. Evolución del rendimiento

In [0]:
%sql
SELECT

anio,

periodo,

ROUND(AVG(nota_final),2) Promedio

FROM rendimiento_estudiantil.gold.vw_monitoreo_riesgo

GROUP BY

anio,

periodo;

5. Estudiantes que requieren intervención

In [0]:
%sql
SELECT

Student_ID,

Major_Subject,

Promedio_Final,

Nivel_Riesgo

FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes

WHERE Nivel_Riesgo='Alto'

ORDER BY Promedio_Final;

## KPIs

Total estudiantes

In [0]:
%sql
SELECT COUNT(*) AS Total_estudiantes

FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes;

Riesgo

In [0]:
%sql
SELECT

Nivel_Riesgo,

COUNT(*) Total

FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes

GROUP BY Nivel_Riesgo;

Promedio General

In [0]:
%sql
SELECT

ROUND(AVG(Promedio_Final),2) AS Promedio

FROM rendimiento_estudiantil.gold.vw_riesgo_estudiantes;

Total matrículas

In [0]:
%sql
SELECT

COUNT(*) AS Total_matriculas

FROM rendimiento_estudiantil.gold.fact_rendimiento_academico;